# 3-Model Benchmark: LightGBM vs XGBoost vs CatBoost

Compares three gradient boosting frameworks on the IEEE-CIS Fraud Detection dataset (V9 engineered features).

**Features used:**
- `train_v9_engineered.pkl` with UID-based aggregations
- Native categorical support where applicable
- Threshold tuning across 0.10–0.50

In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

## Load Data

In [2]:
train = pd.read_pickle('../data/processed/train_v9_engineered.pkl')
drop_cols = ['isFraud', 'TransactionID', 'uid', 'trans_hour']
X = train.drop(columns=drop_cols)
y = train['isFraud']

print(f"Loaded: {train.shape[0]:,} rows x {train.shape[1]} columns")
print(f"Modeling features: {X.shape[1]} columns")
print(f"Target distribution: {y.value_counts().to_dict()}")

Loaded: 590,540 rows x 442 columns
Modeling features: 438 columns
Target distribution: {0: 569877, 1: 20663}


## Train / Test Split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
cat_cols = X_train.select_dtypes(include=['category', 'object']).columns.tolist()
print(f'Train shape: {X_train.shape[0]:,} rows')
print(f'Test shape:  {X_test.shape[0]:,} rows')
print(f'Categorical columns: {len(cat_cols)}')

Train shape: 472,432 rows
Test shape:  118,108 rows
Categorical columns: 31


## Threshold Search Utility

In [4]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]
def threshold_search(model_name, y_true, y_proba):
    rows, best_f1, best_thresh = [], 0.0, 0.25
    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        r = recall_score(y_true, preds)
        p = precision_score(y_true, preds)
        f = f1_score(y_true, preds)
        rows.append({
            'Model': model_name, 'Threshold': t,
            'Recall': round(r, 4), 'Precision': round(p, 4), 'F1': round(f, 4)
        })
        if f > best_f1:
            best_f1, best_thresh = f, t
    auc = roc_auc_score(y_true, y_proba)
    return rows, best_f1, best_thresh, auc

all_results = []

## 1. LightGBM

- Native categorical handling via `category` dtype
- Early stopping (50 rounds)
- `num_leaves=31`, `learning_rate=0.05`

In [5]:
print('Training LightGBM...')
t0 = time.time()
for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=31, random_state=42, n_jobs=-1)
lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], categorical_feature=cat_cols,callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
lgb_proba = lgb_model.predict_proba(X_test)[:, 1]
lgb_rows, lgb_f1, lgb_t, lgb_auc = threshold_search('LightGBM', y_test, lgb_proba)
print(f"LightGBM | Best F1={lgb_f1:.4f} @ threshold={lgb_t} | AUC={lgb_auc:.4f} | {time.time()-t0:.1f}s")
all_results.extend(lgb_rows)

Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.703119 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 41007
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 437
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.0553052
LightGBM | Best F1=0.7669 @ thresh

## 2. XGBoost

- Label-encode categoricals (XGBoost does not natively handle `category` dtype well)
- `tree_method='hist'` for speed
- `max_depth=6`, `subsample=0.8`, `colsample_bytree=0.8`

In [6]:
print('Training XGBoost...')
t0 = time.time()

X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train_xgb[col], X_test_xgb[col]], axis=0).astype(str)
    le.fit(combined)
    X_train_xgb[col] = le.transform(X_train_xgb[col].astype(str))
    X_test_xgb[col] = le.transform(X_test_xgb[col].astype(str))

for df in [X_train_xgb, X_test_xgb]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(-999, inplace=True)  

xgb_model = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method='hist', random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_model.fit(X_train_xgb, y_train, eval_set=[(X_test_xgb, y_test)], verbose=False)

xgb_proba = xgb_model.predict_proba(X_test_xgb)[:, 1]
xgb_rows, xgb_f1, xgb_t, xgb_auc = threshold_search('XGBoost', y_test, xgb_proba)

print(f"XGBoost  | Best F1={xgb_f1:.4f} @ threshold={xgb_t} | AUC={xgb_auc:.4f} | {time.time()-t0:.1f}s")
all_results.extend(xgb_rows)

Training XGBoost...
XGBoost  | Best F1=0.7454 @ threshold=0.25 | AUC=0.9578 | 430.9s


In [7]:
for x in xgb_rows:
    print(x)

{'Model': 'XGBoost', 'Threshold': 0.1, 'Recall': 0.7827, 'Precision': 0.5668, 'F1': 0.6575}
{'Model': 'XGBoost', 'Threshold': 0.15, 'Recall': 0.7392, 'Precision': 0.7054, 'F1': 0.7219}
{'Model': 'XGBoost', 'Threshold': 0.2, 'Recall': 0.7055, 'Precision': 0.7877, 'F1': 0.7444}
{'Model': 'XGBoost', 'Threshold': 0.25, 'Recall': 0.6683, 'Precision': 0.8426, 'F1': 0.7454}
{'Model': 'XGBoost', 'Threshold': 0.3, 'Recall': 0.6424, 'Precision': 0.8791, 'F1': 0.7423}
{'Model': 'XGBoost', 'Threshold': 0.4, 'Recall': 0.587, 'Precision': 0.9238, 'F1': 0.7179}
{'Model': 'XGBoost', 'Threshold': 0.5, 'Recall': 0.5444, 'Precision': 0.947, 'F1': 0.6914}


## 3. CatBoost

- Pass `cat_features` indices explicitly
- `depth=6`, `learning_rate=0.05`
- Early stopping (50 rounds)
- Uncomment `task_type='GPU'` if CUDA is available

In [13]:
print('Training CatBoost...')
t0 = time.time()

cat_features_idx = [X_train.columns.get_loc(c) for c in cat_cols]
X_train_cat = X_train.copy()
X_test_cat = X_test.copy()

for col in cat_cols:
    arr_train = np.array(X_train_cat[col], dtype=object)
    arr_test = np.array(X_test_cat[col], dtype=object)
    
    mask_train = pd.isna(arr_train)
    mask_test = pd.isna(arr_test)
    arr_train[mask_train] = 'missing'
    arr_test[mask_test] = 'missing'
    
    X_train_cat[col] = arr_train
    X_test_cat[col] = arr_test

cat_model = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=8, l2_leaf_reg=3, random_seed=42, verbose=False, early_stopping_rounds=100, grow_policy='Lossguide', task_type='GPU', devices='0')
cat_model.fit(X_train_cat, y_train, cat_features=cat_features_idx, eval_set=(X_test_cat, y_test), verbose=False)

cat_proba = cat_model.predict_proba(X_test_cat)[:, 1]
cat_rows, cat_f1, cat_t, cat_auc = threshold_search('CatBoost', y_test, cat_proba)
print(f"CatBoost | Best F1={cat_f1:.4f} @ threshold={cat_t} | AUC={cat_auc:.4f} | {time.time()-t0:.1f}s")
all_results.extend(cat_rows)

Training CatBoost...
CatBoost | Best F1=0.7881 @ threshold=0.25 | AUC=0.9672 | 429.7s


## Summary

In [14]:
print("=" * 65)
print(f"{'Model':<10} {'Best F1':>8} {'Thresh':>8} {'AUC':>8}")
print("-" * 65)
print(f"{'LightGBM':<10} {lgb_f1:>8.4f} {lgb_t:>8.2f} {lgb_auc:>8.4f}")
print(f"{'XGBoost':<10} {xgb_f1:>8.4f} {xgb_t:>8.2f} {xgb_auc:>8.4f}")
print(f"{'CatBoost':<10} {cat_f1:>8.4f} {cat_t:>8.2f} {cat_auc:>8.4f}")
print("=" * 65)

Model       Best F1   Thresh      AUC
-----------------------------------------------------------------
LightGBM     0.7669     0.25   0.9611
XGBoost      0.7454     0.25   0.9578
CatBoost     0.7881     0.25   0.9672


## All Thresholds

In [18]:
df_results = pd.DataFrame(all_results)
print(df_results.to_string(index=False))

   Model  Threshold  Recall  Precision     F1
LightGBM       0.10  0.7960     0.5986 0.6834
LightGBM       0.15  0.7583     0.7328 0.7453
LightGBM       0.20  0.7259     0.8095 0.7654
LightGBM       0.25  0.6927     0.8590 0.7669
LightGBM       0.30  0.6692     0.8949 0.7658
LightGBM       0.40  0.6148     0.9318 0.7408
LightGBM       0.50  0.5686     0.9537 0.7124
 XGBoost       0.10  0.7827     0.5668 0.6575
 XGBoost       0.15  0.7392     0.7054 0.7219
 XGBoost       0.20  0.7055     0.7877 0.7444
 XGBoost       0.25  0.6683     0.8426 0.7454
 XGBoost       0.30  0.6424     0.8791 0.7423
 XGBoost       0.40  0.5870     0.9238 0.7179
 XGBoost       0.50  0.5444     0.9470 0.6914
CatBoost       0.10  0.6934     0.4991 0.5805
CatBoost       0.15  0.6477     0.6480 0.6479
CatBoost       0.20  0.6080     0.7528 0.6727
CatBoost       0.25  0.5734     0.8170 0.6739
CatBoost       0.30  0.5504     0.8595 0.6711
CatBoost       0.40  0.5110     0.9135 0.6554
CatBoost       0.50  0.4728     0.